# Trabajo 2: Datos Base — Sistema de Recomendación por Géneros

**Asignatura**: Sistemas Recomendadores — MIARFID  
**Objetivo**: Construir un vector de preferencias de género por usuario a partir del dataset MovieLens.

Cada usuario tendrá un vector de 20 posiciones (una por género) con un ratio de preferencia normalizado.  
A la hora de recomendar, se filtrarán las **n posiciones más altas** del vector.

---

## 0. Instalación de dependencias e imports

Ejecuta esta celda para instalar las librerías necesarias (solo la primera vez).

In [ ]:
# Instalar dependencias si no están disponibles
import subprocess, sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

for pkg in ["pandas", "numpy", "matplotlib", "seaborn"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Instalando {pkg}...")
        install(pkg)

print("✅ Todas las dependencias están instaladas.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
sns.set_palette('husl')

# Ruta base del dataset
DATA_DIR = Path('Data_set')

print(f"📁 Directorio de datos: {DATA_DIR.resolve()}")
print(f"📦 Archivos disponibles: {[f.name for f in DATA_DIR.iterdir() if f.is_file()]}")

---
## Fase 1: Carga y Limpieza de Datos

### 1.1 Carga de los ficheros CSV

In [ ]:
# --- Géneros ---
df_generos = pd.read_csv(DATA_DIR / 'generos.csv', sep=';', encoding='latin-1')
print(f"🎭 Géneros cargados: {len(df_generos)}")
df_generos

In [ ]:
# --- Películas ---
df_peliculas = pd.read_csv(DATA_DIR / 'peliculas.csv', sep=';', encoding='latin-1', decimal=',')
print(f"🎬 Películas cargadas: {len(df_peliculas)}")
print(f"   Columnas: {list(df_peliculas.columns)}")
df_peliculas.head()

In [ ]:
# --- Ratings ---
df_ratings = pd.read_csv(DATA_DIR / 'ratings_small.csv', sep=';', decimal=',')
print(f"⭐ Ratings cargados: {len(df_ratings)}")
print(f"   Usuarios únicos: {df_ratings['userId'].nunique()}")
print(f"   Películas únicas: {df_ratings['movieId'].nunique()}")
df_ratings.head()

In [ ]:
# --- Keywords ---
df_keywords = pd.read_csv(DATA_DIR / 'keywords.csv', sep=';', encoding='latin-1')
print(f"🔑 Keywords cargadas: {len(df_keywords)} películas")
df_keywords.head(3)

In [ ]:
# --- Links ---
df_links = pd.read_csv(DATA_DIR / 'links.csv', sep=';')
print(f"🔗 Links cargados: {len(df_links)} películas")
df_links.head(3)

### 1.2 Mapeo Película → Géneros

El archivo `peliculas.csv` tiene las columnas de géneros como `id_genero` repetidas. Necesitamos extraer los IDs de género por cada película.

In [ ]:
# Las columnas de género son todas las llamadas 'id_genero' (repetidas)
# Extraemos los IDs de género para cada película
genero_cols = [col for col in df_peliculas.columns if col.startswith('id_genero')]
print(f"Columnas de género encontradas: {genero_cols}")

# Crear un diccionario: id_pelicula -> lista de ids de género
pelicula_generos = {}

for _, row in df_peliculas.iterrows():
    movie_id = row['id']
    generos = []
    for col in genero_cols:
        val = row[col]
        if pd.notna(val) and val != '':
            try:
                generos.append(int(float(val)))
            except (ValueError, TypeError):
                pass
    pelicula_generos[movie_id] = generos

# Crear mapeo IdDataset -> id interno del género
id_dataset_to_genre_idx = dict(zip(df_generos['IdDataset'], df_generos['id']))
id_dataset_to_genre_name = dict(zip(df_generos['IdDataset'], df_generos['GeneroSP']))
genre_idx_to_name = dict(zip(df_generos['id'], df_generos['GeneroSP']))
genre_names = [genre_idx_to_name[i] for i in range(len(df_generos))]

print(f"\n📊 Estadísticas del mapeo:")
n_generos_por_pelicula = [len(v) for v in pelicula_generos.values()]
print(f"   Media de géneros por película: {np.mean(n_generos_por_pelicula):.2f}")
print(f"   Máx géneros por película: {np.max(n_generos_por_pelicula)}")
print(f"   Películas sin género: {sum(1 for v in pelicula_generos.values() if len(v) == 0)}")
print(f"\n🎭 Lista de géneros: {genre_names}")

### 1.3 Validación de integridad

In [ ]:
# Verificar que todos los movieId en ratings existen en películas
movies_in_ratings = set(df_ratings['movieId'].unique())
movies_in_peliculas = set(df_peliculas['id'].unique())

missing_movies = movies_in_ratings - movies_in_peliculas
print(f"📋 Películas en ratings que NO están en peliculas.csv: {len(missing_movies)}")
if missing_movies:
    print(f"   Ejemplo de IDs faltantes: {list(missing_movies)[:10]}")
    print(f"   ⚠️ Estos ratings serán ignorados al no tener información de género.")

# Ratings con información de género válida
df_ratings_valid = df_ratings[df_ratings['movieId'].isin(movies_in_peliculas)].copy()
print(f"\n✅ Ratings válidos: {len(df_ratings_valid)} / {len(df_ratings)} ({100*len(df_ratings_valid)/len(df_ratings):.1f}%)")
print(f"   Usuarios: {df_ratings_valid['userId'].nunique()}")
print(f"   Películas: {df_ratings_valid['movieId'].nunique()}")

### 1.4 Estadísticas descriptivas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribución de ratings
axes[0].hist(df_ratings_valid['rating'], bins=10, color='#4ECDC4', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribución de Ratings', fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(df_ratings_valid['rating'].mean(), color='red', linestyle='--', label=f"Media: {df_ratings_valid['rating'].mean():.2f}")
axes[0].legend()

# Ratings por usuario
ratings_per_user = df_ratings_valid.groupby('userId').size()
axes[1].hist(ratings_per_user, bins=50, color='#FF6B6B', edgecolor='white', alpha=0.8)
axes[1].set_title('Ratings por Usuario', fontweight='bold')
axes[1].set_xlabel('Nº de Ratings')
axes[1].set_ylabel('Nº de Usuarios')
axes[1].axvline(ratings_per_user.mean(), color='blue', linestyle='--', label=f"Media: {ratings_per_user.mean():.0f}")
axes[1].legend()

# Distribución de géneros en el catálogo
genre_counts = {}
for glist in pelicula_generos.values():
    for g in glist:
        name = id_dataset_to_genre_name.get(g, f'Desconocido({g})')
        genre_counts[name] = genre_counts.get(name, 0) + 1
genre_series = pd.Series(genre_counts).sort_values(ascending=True)
genre_series.plot(kind='barh', ax=axes[2], color='#45B7D1', edgecolor='white')
axes[2].set_title('Distribución de Géneros en Catálogo', fontweight='bold')
axes[2].set_xlabel('Nº de Películas')

plt.tight_layout()
plt.show()

---
## Fase 2: Construcción del Vector de Preferencias por Usuario

**Estrategia**: Distribución ponderada del rating entre los géneros de cada película.

Para cada película valorada por un usuario con rating $r$ y que tiene $k$ géneros:
- Cada género de esa película recibe una contribución de $\frac{r}{k}$

El ratio final de preferencia es la **media** de estas contribuciones para cada género:

$$\text{Preferencia}(u, g) = \frac{\sum_{i \in \text{peliculas\_con\_g}} \frac{r_i}{k_i}}{\text{count}(\text{peliculas\_con\_g})}$$

In [ ]:
NUM_GENRES = len(df_generos)
users = sorted(df_ratings_valid['userId'].unique())
NUM_USERS = len(users)

print(f"🔨 Construyendo matriz de preferencias: {NUM_USERS} usuarios × {NUM_GENRES} géneros...")

# Inicializar matrices para acumular sumas y conteos
# user_genre_sum[u][g] = suma de contribuciones ponderadas
# user_genre_count[u][g] = número de películas que contribuyen
user_idx_map = {uid: idx for idx, uid in enumerate(users)}

user_genre_sum = np.zeros((NUM_USERS, NUM_GENRES))
user_genre_count = np.zeros((NUM_USERS, NUM_GENRES))

skipped = 0
processed = 0

for _, row in df_ratings_valid.iterrows():
    user_id = row['userId']
    movie_id = row['movieId']
    rating = row['rating']
    
    u_idx = user_idx_map[user_id]
    
    # Obtener géneros de esta película
    generos_pelicula = pelicula_generos.get(movie_id, [])
    
    if not generos_pelicula:
        skipped += 1
        continue
    
    k = len(generos_pelicula)  # número de géneros de esta película
    
    for g_dataset_id in generos_pelicula:
        g_idx = id_dataset_to_genre_idx.get(g_dataset_id)
        if g_idx is not None:
            user_genre_sum[u_idx, g_idx] += rating / k  # contribución ponderada
            user_genre_count[u_idx, g_idx] += 1
    
    processed += 1

print(f"   ✅ Procesados: {processed} ratings")
print(f"   ⚠️ Saltados (sin género): {skipped}")

In [ ]:
# Calcular la media: ratio = suma / count
# Donde count > 0, calculamos la media; donde count == 0, dejamos NaN
with np.errstate(divide='ignore', invalid='ignore'):
    user_genre_matrix = np.where(
        user_genre_count > 0,
        user_genre_sum / user_genre_count,
        np.nan
    )

# Crear DataFrame con la matriz
df_preferences = pd.DataFrame(
    user_genre_matrix,
    index=users,
    columns=genre_names
)
df_preferences.index.name = 'userId'

print(f"📊 Matriz de preferencias creada: {df_preferences.shape}")
print(f"\n--- Primeros 10 usuarios ---")
df_preferences.head(10).round(3)

In [ ]:
# Estadísticas de cobertura (posiciones rellenas por usuario)
cobertura = df_preferences.notna().sum(axis=1)

print(f"\n📈 Cobertura de géneros por usuario:")
print(f"   Media de géneros cubiertos: {cobertura.mean():.1f} / {NUM_GENRES}")
print(f"   Mediana: {cobertura.median():.0f}")
print(f"   Mínimo: {cobertura.min()} | Máximo: {cobertura.max()}")
print(f"   Usuarios con cobertura completa (20/20): {(cobertura == NUM_GENRES).sum()}")
print(f"   Usuarios con ≥15 géneros: {(cobertura >= 15).sum()}")

---
## Fase 3: Normalización del Vector

Normalizamos los vectores de preferencia al rango **[0, 1]** usando Min-Max Scaling por usuario.  
Las posiciones sin información (`NaN`) se mantienen como `NaN`.

In [ ]:
# Normalización Min-Max por usuario (fila)
def normalize_row(row):
    valid = row.dropna()
    if len(valid) == 0:
        return row
    min_val = valid.min()
    max_val = valid.max()
    if max_val == min_val:
        # Todos los valores son iguales -> normalizar a 0.5
        return row.where(row.isna(), 0.5)
    return (row - min_val) / (max_val - min_val)

df_preferences_norm = df_preferences.apply(normalize_row, axis=1)

print("📊 Matriz normalizada [0, 1]:")
df_preferences_norm.head(10).round(3)

In [ ]:
# También guardamos la versión sin normalizar para análisis
print("\n📊 Comparación de un usuario ejemplo (primero disponible):")
ejemplo_user = users[0]
comp = pd.DataFrame({
    'Preferencia_Raw': df_preferences.loc[ejemplo_user].round(3),
    'Preferencia_Norm': df_preferences_norm.loc[ejemplo_user].round(3),
    'Nº_Películas': user_genre_count[user_idx_map[ejemplo_user]]
})
comp = comp.dropna(subset=['Preferencia_Raw'])
print(f"\nUsuario {ejemplo_user}:")
comp

---
## Fase 4: Análisis Exploratorio y Validación

### 4.1 Heatmap de la Matriz de Preferencias

In [ ]:
# Heatmap de una muestra de usuarios
sample_users = users[:50]  # Primeros 50 usuarios
sample_matrix = df_preferences_norm.loc[sample_users]

fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(
    sample_matrix,
    cmap='YlOrRd',
    annot=False,
    xticklabels=True,
    yticklabels=True,
    cbar_kws={'label': 'Preferencia Normalizada'},
    linewidths=0.1,
    linecolor='white',
    ax=ax,
    mask=sample_matrix.isna()  # Mostrar NaN como vacío
)
ax.set_title('Mapa de Preferencias de Género (Primeros 50 Usuarios)', fontsize=16, fontweight='bold')
ax.set_xlabel('Género', fontsize=12)
ax.set_ylabel('Usuario ID', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 4.2 Distribución de Cobertura

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Distribución de cobertura
axes[0].hist(cobertura, bins=range(0, NUM_GENRES + 2), color='#6C5CE7', edgecolor='white', alpha=0.8, align='left')
axes[0].set_title('Distribución de Cobertura de Géneros por Usuario', fontweight='bold')
axes[0].set_xlabel('Nº de Géneros con Información')
axes[0].set_ylabel('Nº de Usuarios')
axes[0].axvline(cobertura.mean(), color='red', linestyle='--', label=f"Media: {cobertura.mean():.1f}")
axes[0].legend()
axes[0].set_xticks(range(0, NUM_GENRES + 1, 2))

# Relación entre nº de ratings y cobertura
ratings_count = df_ratings_valid.groupby('userId').size()
scatter_data = pd.DataFrame({'ratings': ratings_count, 'cobertura': cobertura})
axes[1].scatter(scatter_data['ratings'], scatter_data['cobertura'], alpha=0.5, s=20, color='#00B894')
axes[1].set_title('Relación: Nº Ratings vs Cobertura de Géneros', fontweight='bold')
axes[1].set_xlabel('Nº de Ratings del Usuario')
axes[1].set_ylabel('Nº de Géneros Cubiertos')
axes[1].axhline(NUM_GENRES, color='red', linestyle='--', alpha=0.5, label='Máximo (20 géneros)')
axes[1].legend()

plt.tight_layout()
plt.show()

### 4.3 Preferencias Globales y Top Géneros

In [ ]:
# Preferencia media global por género
global_pref = df_preferences.mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Preferencia media
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(global_pref)))
global_pref.plot(kind='barh', ax=axes[0], color=colors[::-1], edgecolor='white')
axes[0].set_title('Preferencia Media por Género (todos los usuarios)', fontweight='bold')
axes[0].set_xlabel('Rating Medio Ponderado')
axes[0].invert_yaxis()

# Porcentaje de usuarios que tienen cada género
genre_popularity = (df_preferences.notna().sum() / NUM_USERS * 100).sort_values(ascending=False)
genre_popularity.plot(kind='barh', ax=axes[1], color='#FDCB6E', edgecolor='white')
axes[1].set_title('% de Usuarios con cada Género en su Perfil', fontweight='bold')
axes[1].set_xlabel('% de Usuarios')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

### 4.4 Perfil de Usuarios Ejemplo

In [ ]:
# Seleccionar 4 usuarios representativos
# Uno con pocos ratings, uno con muchos, y dos intermedios
user_rating_counts = df_ratings_valid.groupby('userId').size().sort_values()
example_users = [
    user_rating_counts.index[0],         # Menos ratings
    user_rating_counts.index[len(user_rating_counts)//3],  # Tercil inferior
    user_rating_counts.index[2*len(user_rating_counts)//3],  # Tercil superior
    user_rating_counts.index[-1]          # Más ratings
]

fig, axes = plt.subplots(2, 2, figsize=(18, 12), subplot_kw=dict(polar=True))
axes = axes.flatten()

for idx, uid in enumerate(example_users):
    ax = axes[idx]
    n_ratings = user_rating_counts[uid]
    
    values = df_preferences_norm.loc[uid].fillna(0).values
    categories = genre_names
    N = len(categories)
    
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    values_plot = np.concatenate([values, [values[0]]])
    angles += angles[:1]
    
    ax.fill(angles, values_plot, alpha=0.25, color=f'C{idx}')
    ax.plot(angles, values_plot, linewidth=2, color=f'C{idx}')
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=8)
    ax.set_title(f'Usuario {uid} ({n_ratings} ratings)', fontweight='bold', size=13, pad=20)
    ax.set_ylim(0, 1)

plt.suptitle('Perfiles de Preferencia de 4 Usuarios Representativos', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 4.5 Correlación entre Géneros

In [ ]:
# Matriz de correlación entre las preferencias de los géneros
corr_matrix = df_preferences.corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8, 'label': 'Correlación'},
    ax=ax,
    vmin=-1, vmax=1
)
ax.set_title('Correlación entre Preferencias de Géneros', fontsize=16, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 4.6 Validación Manual

Verificamos el cálculo del vector para un usuario concreto trazando manualmente sus ratings.

In [ ]:
# Validación manual para userId = 1
test_user = users[0]
print(f"🔍 Validación manual para Usuario {test_user}")
print("=" * 70)

user_ratings = df_ratings_valid[df_ratings_valid['userId'] == test_user]
print(f"\nPelículas valoradas: {len(user_ratings)}")

# Mostrar detalle de cada película y sus géneros
manual_sums = np.zeros(NUM_GENRES)
manual_counts = np.zeros(NUM_GENRES)

for _, row in user_ratings.iterrows():
    movie_id = row['movieId']
    rating = row['rating']
    generos = pelicula_generos.get(movie_id, [])
    
    if not generos:
        continue
    
    genre_names_this = [id_dataset_to_genre_name.get(g, '?') for g in generos]
    k = len(generos)
    
    # Buscar título
    titulo_row = df_peliculas[df_peliculas['id'] == movie_id]
    titulo = titulo_row['titulo'].values[0] if len(titulo_row) > 0 else '?'
    
    print(f"  📽️ {titulo} (id={movie_id}) | Rating: {rating} | Géneros ({k}): {genre_names_this}")
    
    for g_dataset_id in generos:
        g_idx = id_dataset_to_genre_idx.get(g_dataset_id)
        if g_idx is not None:
            manual_sums[g_idx] += rating / k
            manual_counts[g_idx] += 1

print("\n📊 Vector calculado manualmente vs. programáticamente:")
for g_idx in range(NUM_GENRES):
    if manual_counts[g_idx] > 0:
        manual_val = manual_sums[g_idx] / manual_counts[g_idx]
        prog_val = df_preferences.loc[test_user].iloc[g_idx]
        match = '✅' if abs(manual_val - prog_val) < 0.001 else '❌'
        print(f"  {match} {genre_names[g_idx]:20s}: Manual={manual_val:.3f} | Programa={prog_val:.3f} | Películas={int(manual_counts[g_idx])}")

---
## Fase 5: Filtro de Recomendación Básico

Implementamos una función de recomendación que:
1. Toma las **n posiciones más altas** del vector del usuario
2. Busca películas no vistas que pertenezcan a esos géneros
3. Las ordena por su puntuación media en el dataset

In [ ]:
def recomendar(user_id, n_top_genres=5, n_recomendaciones=10, df_pref=df_preferences_norm):
    """
    Recomienda películas a un usuario basándose en sus géneros preferidos.
    
    Args:
        user_id: ID del usuario
        n_top_genres: Número de géneros top a considerar
        n_recomendaciones: Número de películas a recomendar
        df_pref: DataFrame de preferencias (normalizado)
    
    Returns:
        DataFrame con las películas recomendadas
    """
    # 1. Obtener vector de preferencias del usuario
    user_prefs = df_pref.loc[user_id].dropna()
    
    if len(user_prefs) == 0:
        print(f"⚠️ El usuario {user_id} no tiene preferencias calculadas.")
        return pd.DataFrame()
    
    # 2. Seleccionar los n géneros con mayor preferencia
    top_genres = user_prefs.nlargest(n_top_genres)
    top_genre_names = top_genres.index.tolist()
    
    # Mapear nombres de género de vuelta a IDs de dataset
    genre_name_to_dataset_id = dict(zip(df_generos['GeneroSP'], df_generos['IdDataset']))
    top_genre_dataset_ids = set([genre_name_to_dataset_id[name] for name in top_genre_names if name in genre_name_to_dataset_id])
    
    # 3. Películas ya vistas por el usuario
    peliculas_vistas = set(df_ratings_valid[df_ratings_valid['userId'] == user_id]['movieId'].values)
    
    # 4. Buscar películas no vistas con los géneros preferidos
    candidatas = []
    for movie_id, generos in pelicula_generos.items():
        if movie_id in peliculas_vistas:
            continue
        # ¿Tiene alguno de los géneros top?
        matching_genres = set(generos) & top_genre_dataset_ids
        if matching_genres:
            # Score: proporción de géneros top que tiene
            score = len(matching_genres) / n_top_genres
            candidatas.append((movie_id, score, len(matching_genres)))
    
    if not candidatas:
        print("⚠️ No se encontraron candidatas.")
        return pd.DataFrame()
    
    # 5. Ordenar por score y puntuación media
    df_candidatas = pd.DataFrame(candidatas, columns=['movieId', 'score_genero', 'n_matching'])
    df_candidatas = df_candidatas.merge(
        df_peliculas[['id', 'titulo', 'puntuacion_media', 'votos']],
        left_on='movieId', right_on='id', how='left'
    )
    
    # Score combinado: afinidad de género + puntuación media normalizada
    max_punt = df_candidatas['puntuacion_media'].max() if df_candidatas['puntuacion_media'].max() > 0 else 1
    df_candidatas['score_final'] = (
        0.6 * df_candidatas['score_genero'] + 
        0.4 * (df_candidatas['puntuacion_media'] / max_punt)
    )
    
    # Filtrar películas con suficientes votos
    df_candidatas = df_candidatas[df_candidatas['votos'] >= 50]
    
    resultado = df_candidatas.nlargest(n_recomendaciones, 'score_final')[
        ['titulo', 'puntuacion_media', 'votos', 'n_matching', 'score_final']
    ].reset_index(drop=True)
    resultado.index += 1  # ranking desde 1
    
    return resultado, top_genres

In [ ]:
# Ejemplo de recomendación
test_user_rec = users[3]  # Tomamos un usuario con bastantes ratings
n_top = 5

recomendaciones, top_genres = recomendar(test_user_rec, n_top_genres=n_top, n_recomendaciones=15)

print(f"🎯 Recomendaciones para Usuario {test_user_rec}")
print(f"\n🏆 Top {n_top} géneros preferidos:")
for genre, score in top_genres.items():
    print(f"   {genre}: {score:.3f}")

print(f"\n🎬 Top 15 Películas Recomendadas:")
recomendaciones

In [ ]:
# Visualizar el perfil del usuario y sus recomendaciones
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Perfil del usuario
user_profile = df_preferences_norm.loc[test_user_rec].dropna().sort_values(ascending=True)
colors_profile = ['#FF6B6B' if name in top_genres.index else '#45B7D1' for name in user_profile.index]
user_profile.plot(kind='barh', ax=axes[0], color=colors_profile, edgecolor='white')
axes[0].set_title(f'Perfil de Preferencias — Usuario {test_user_rec}', fontweight='bold')
axes[0].set_xlabel('Preferencia Normalizada')
axes[0].legend(['Género Top seleccionado', 'Otros géneros'], loc='lower right')

# Scores de las recomendaciones
if len(recomendaciones) > 0:
    rec_display = recomendaciones.head(10)
    y_pos = range(len(rec_display))
    axes[1].barh(y_pos, rec_display['score_final'], color='#6C5CE7', edgecolor='white', alpha=0.8)
    axes[1].set_yticks(y_pos)
    # Truncar títulos largos
    labels = [t[:40] + '...' if len(str(t)) > 40 else str(t) for t in rec_display['titulo']]
    axes[1].set_yticklabels(labels)
    axes[1].set_title('Top 10 Películas Recomendadas', fontweight='bold')
    axes[1].set_xlabel('Score Final')
    axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

---
## Fase 6: Exportación de Resultados

In [ ]:
# Crear directorio de resultados
results_dir = Path('resultados')
results_dir.mkdir(exist_ok=True)

# Exportar matriz de preferencias raw
df_preferences.to_csv(results_dir / 'user_genre_matrix_raw.csv', sep=';', decimal=',')
print(f"✅ Matriz raw guardada en: {results_dir / 'user_genre_matrix_raw.csv'}")

# Exportar matriz normalizada
df_preferences_norm.to_csv(results_dir / 'user_genre_matrix_normalized.csv', sep=';', decimal=',')
print(f"✅ Matriz normalizada guardada en: {results_dir / 'user_genre_matrix_normalized.csv'}")

# Exportar conteos (cuántas películas contribuyen a cada género por usuario)
df_counts = pd.DataFrame(
    user_genre_count,
    index=users,
    columns=genre_names
)
df_counts.index.name = 'userId'
df_counts.to_csv(results_dir / 'user_genre_counts.csv', sep=';')
print(f"✅ Conteos guardados en: {results_dir / 'user_genre_counts.csv'}")

---
## Resumen y Conclusiones

### Resultados clave

In [ ]:
print("=" * 70)
print("📋 RESUMEN DEL TRABAJO 2: DATOS BASE")
print("=" * 70)
print(f"\n📦 Dataset:")
print(f"   • {len(df_peliculas)} películas en catálogo")
print(f"   • {NUM_GENRES} géneros")
print(f"   • {len(df_ratings)} ratings totales")
print(f"   • {len(df_ratings_valid)} ratings válidos (con info de género)")
print(f"   • {NUM_USERS} usuarios")

print(f"\n📊 Matriz de Preferencias:")
print(f"   • Dimensiones: {df_preferences.shape[0]} × {df_preferences.shape[1]}")
print(f"   • Método: Distribución ponderada del rating entre géneros")
print(f"   • Normalización: Min-Max [0, 1] por usuario")

print(f"\n📈 Cobertura:")
print(f"   • Media de géneros cubiertos/usuario: {cobertura.mean():.1f} / {NUM_GENRES}")
print(f"   • Usuarios con cobertura completa: {(cobertura == NUM_GENRES).sum()} ({100*(cobertura == NUM_GENRES).sum()/NUM_USERS:.1f}%)")
print(f"   • Usuarios con ≥15 géneros: {(cobertura >= 15).sum()} ({100*(cobertura >= 15).sum()/NUM_USERS:.1f}%)")

print(f"\n🏆 Top 5 géneros más populares:")
for name, pct in genre_popularity.head().items():
    print(f"   • {name}: {pct:.1f}% de usuarios")

print(f"\n💾 Archivos exportados en: {results_dir.resolve()}")
print("=" * 70)